In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('val_bpb.csv')
df.head()

,config,seed,jobid,final_step,final_val_bpb,file
0,baseline,1,19490487,2520,0.884042,nanochat-baseline-seed1.o19490487
1,baseline,2,19490488,2520,0.883857,nanochat-baseline-seed2.o19490488
2,baseline,3,19490489,2520,0.883472,nanochat-baseline-seed3.o19490489
3,baseline,4,19490490,2520,0.884666,nanochat-baseline-seed4.o19490490
4,baseline,5,19490491,2520,0.883770,nanochat-baseline-seed5.o19490491


In [3]:
df.groupby('config').describe()['final_val_bpb']

,count,mean,std,min,25%,50%,75%,max
config,,,,,,,,
baseline,10.0,0.884107,0.000482,0.883472,0.883862,0.883987,0.884205,0.885151
dropout01,10.0,0.878600,0.000630,0.877943,0.878070,0.878493,0.878820,0.879774
dropout03,10.0,0.872357,0.000559,0.871718,0.872010,0.872224,0.872356,0.873365
dropout05,10.0,0.870093,0.000502,0.869000,0.869888,0.870169,0.870383,0.870830
ve_gate_const_momentum,10.0,0.884077,0.000738,0.882600,0.883820,0.883953,0.884484,0.885422
ve_gate_momentum_scheduling,10.0,0.884076,0.000497,0.883537,0.883665,0.884057,0.884268,0.885206
ve_gate_relu,10.0,0.884939,0.001362,0.883229,0.883985,0.884488,0.885941,0.887123


In [7]:
df.groupby('config').describe()['final_val_bpb'] - df.groupby('config').describe().loc['baseline', 'final_val_bpb']

,count,mean,std,min,25%,50%,75%,max
config,,,,,,,,
baseline,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
dropout01,0.0,-0.005507,0.000148,-0.005529,-0.005792,-0.005494,-0.005385,-0.005377
dropout03,0.0,-0.011749,0.000077,-0.011754,-0.011852,-0.011763,-0.011849,-0.011786
dropout05,0.0,-0.014013,0.000020,-0.014472,-0.013974,-0.013818,-0.013822,-0.014321
ve_gate_const_momentum,0.0,-0.000029,0.000256,-0.000872,-0.000042,-0.000034,0.000279,0.000271
ve_gate_momentum_scheduling,0.0,-0.000031,0.000015,0.000065,-0.000198,0.000070,0.000063,0.000055
ve_gate_relu,0.0,0.000832,0.000880,-0.000243,0.000123,0.000501,0.001736,0.001972


In [4]:
final_val_bpbs = {}

for config in ['baseline', 'dropout05', 'dropout03', 'dropout01', 've_gate_const_momentum', 've_gate_momentum_scheduling', 've_gate_relu']:
    final_val_bpbs[config] = df.loc[df['config'] == config, 'final_val_bpb']

In [5]:
import numpy as np
from scipy.stats import ttest_ind

def compute_cohen_d(group1, group2):
    """Computes Cohen's d for two independent samples."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    s_pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    return (np.mean(group1) - np.mean(group2)) / s_pooled

configs = ['dropout05', 'dropout03', 'dropout01', 've_gate_const_momentum', 've_gate_momentum_scheduling', 've_gate_relu']

for config in configs:
    baseline_data = final_val_bpbs['baseline']
    config_data = final_val_bpbs[config]
    
    print(f'--- {config} ---')
    
    res_gr = ttest_ind(baseline_data, config_data, equal_var=False, alternative='greater')
    
    d = compute_cohen_d(baseline_data, config_data)
    
    print(f'    p-value (greater):   {res_gr.pvalue:.4e}')
    print(f'    Cohen\'s d:          {d:.4f}')
    
    if d > 0.8:
        strength = "Large"
    elif d > 0.5:
        strength = "Medium"
    elif d > 0.2:
        strength = "Small"
    else:
        strength = "Negligible"
        
    print(f'    Effect Magnitude:    {strength}\n')

--- dropout05 ---
    p-value (greater):   6.4370e-23
    Cohen's d:          28.4689
    Effect Magnitude:    Large

--- dropout03 ---
    p-value (greater):   8.7059e-21
    Cohen's d:          22.5097
    Effect Magnitude:    Large

--- dropout01 ---
    p-value (greater):   3.8647e-14
    Cohen's d:          9.8225
    Effect Magnitude:    Large

--- ve_gate_const_momentum ---
    p-value (greater):   4.5853e-01
    Cohen's d:          0.0473
    Effect Magnitude:    Negligible

--- ve_gate_momentum_scheduling ---
    p-value (greater):   4.4485e-01
    Cohen's d:          0.0629
    Effect Magnitude:    Negligible

--- ve_gate_relu ---
    p-value (greater):   9.5231e-01
    Cohen's d:          -0.8143
    Effect Magnitude:    Negligible

